# Setup

## Imports

In [ ]:
# Native packages
from pathlib import Path
from importlib import reload
import datetime as dt
import numpy as np
import subprocess
import itertools
import difflib

# Other packages
import matplotlib.pyplot as plt
import metpy.calc as mpc
import jinja2
import pandas as pd
from metpy.units import units

# My packages
import ps
import ps.projects.thermals as pst
import carlee_tools as ct
import xarrams as xrr

plt.style.use("cd_slides")

## RAMSIN diffing

In [ ]:
ct.recursive_reload(xrr)


# ---------------------------------------------------------------------------
# Isolate the RAMSIN change for each parameter individually.
#
# Take the first parameter set as the base. For each parameter, find the
# experiment whose parameter set differs from the base in *exactly* that one
# parameter, and diff its RAMSIN (and submit script) against the base. This
# isolates the effect of each parameter on the rendered files so we can confirm
# each one is wired up correctly.
# ---------------------------------------------------------------------------
def diff_files(path_a, path_b):
    return "\n".join(
        difflib.unified_diff(
            path_a.read_text().splitlines(),
            path_b.read_text().splitlines(),
            fromfile=str(path_a),
            tofile=str(path_b),
        )
    )


def find_single_param_variant(parameter_sets, base_pset, param):
    """Return the experiment name whose pset differs from `base_pset` in only `param`."""
    for name, pset in parameter_sets.items():
        differing = [k for k in base_pset if pset[k] != base_pset[k]]
        if differing == [param]:
            return name
    return None


def diff_ramsins_by_parameter(experiment_ramsins):
    # Use the first parameter set as the base. Recover its parameters from the
    # experiment name with ct.utils.to_kv_pairs (the inverse of ct.to_kv_str).
    experiment_parameter_sets = {
        k: ct.kv.to_kv_pairs(k, parse_floats=True)
        for k in experiment_ramsins.keys()
    }
    base_experiment, base_pset = ct.first_item(experiment_parameter_sets)
    print(f"Base experiment: {base_experiment}")
    print(f"Base parameters: {base_pset}\n")

    for param in base_pset:
        match = find_single_param_variant(
            experiment_parameter_sets, base_pset, param
        )
        if match is None:
            print(
                f"=== '{param}': no single-parameter variant available,"
                " skipping ===\n"
            )
            continue
        match_pset = experiment_parameter_sets[match]
        print(
            f"=== Isolating '{param}': {base_pset[param]} ->"
            f" {match_pset[param]} ==="
        )
        print(f"  base:  {base_experiment}")
        print(f"  match: {match}")
        print("RAMSIN diff:")
        print(
            diff_files(
                experiment_ramsins[base_experiment], experiment_ramsins[match]
            )
        )
        # print("Submit script diff:")
        # print(
        #     diff_files(
        #         experiment_submit_scripts[base_experiment],
        #         experiment_submit_scripts[match],
        #     )
        # )
        print("")

# Validation

In [ ]:
# Base state of the squall-line run, whose sounding (TS/PS/RTS) we are copying
# into the new supercell run.  We need its hydrostatic P(z) to convert the
# sounding's pressure levels into heights.
sl_bs_ds = xrr.read_rams_output(
    Path(
        "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/paper_base_state/output"
    ).glob("a-A-*.h5")
)
sl_bs_ds = xrr.calculate_thermodynamic_variables(sl_bs_ds)
sl_bs_profile = sl_bs_ds.mean(["x", "y"]).isel(time=0, drop=True).squeeze()
sl_bs_p = sl_bs_profile["P"].values  # hPa, decreasing with height
sl_bs_z = sl_bs_profile["z"].values  # m, increasing


In [ ]:
SLURM_QUEUE = "batch_short"

N_NODES = 4

SUBMIT_SLURM_JOBS = False

experiment_subgroup_dir = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sc_wk/validation"
)

# Only things we're changing besides paths is litefile output (off) and
# the sounding in the validation case
sc_ramsin_template_path = Path(
    "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sc_wk/RAMSIN.newtracer_forcing-grant2014_dx-500_frqlite-2_stratwinds"
)

RAMS_EXECUTABLE_PATH = Path(
    "/autofs/nccsopen-svm1_home/cmdavis4/projects/tracer_paper/RAMS_tracer_source/bin.rams/rams-6.3.04_tracer_source"
)

ramsins_paths = {}

for fixed in [True]:

    experiment_name = "sl_profile"
    experiment_dir = experiment_subgroup_dir / experiment_name
    experiment_dir.mkdir(exist_ok=True, parents=False)

    sl_ramsin_path = Path(
        "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sl_wk/RAMSIN.newtracer_blspinup-no_forcing-infinitecoldbubble4_deltax-500_frqlite-2"
    )
    # Read in the sounding from the squall line template
    # sl_sounding_df = xrr.sounding_df_from_ramsin(sl_ramsin_path)
    # display(sl_sounding_df.head())
    # sounding_dict = xrr.sounding_df_to_ramsin_dict(sl_sounding_df)
    # Keep only the keys we want
    # sounding_dict = {k: v for k, v in sounding_dict.items() if k in ["TS", "PS", "RTS"]}
    # print(sounding_dict.keys())

    copy_vars = ["TS", "PS", "RTS"]
    copy_flags = ["IPSFLG", "ITSFLG", "IRTSFLG"]
    # Also get the flag keys
    ramsin_params = (
        {"FRQLITE": 0}
        | {
            field: xrr.to_ramsin_values_str(
                xrr.soundings._parse_ramsin_sounding_array(
                    sl_ramsin_path.read_text(), field
                )
            )
            for field in copy_vars
        }
        | {
            field: xrr.execution._parse_ramsin_field(
                sl_ramsin_path.read_text(), field
            )
            for field in copy_flags
        }
    )

    # ------------------------------------------------------------------
    # Fix the winds.
    #
    # The supercell template's sounding is specified on *height* levels
    # (IPSFLG=1), so its US/VS are a U(z) profile.  The squall-line sounding we
    # are copying in is specified on *pressure* levels (IPSFLG=0), so we have to
    # re-express that same U(z) on the new pressure levels.
    #
    # Interpolating U directly from pressure to pressure is wrong: the two runs
    # have different thermodynamic profiles (and different surface pressures --
    # 1000 hPa vs 1013.25 hPa), so a given pressure sits at a different height in
    # each.  Instead:
    #   1. map each new sounding pressure level to a height, using the P(z) of
    #      the squall-line base state (that is the thermodynamic profile the new
    #      run will actually have), and
    #   2. read U and V off the supercell base state at those heights.
    # ------------------------------------------------------------------
    def interp_with_linear_extrapolation(x_new, x, y):
        """np.interp, but extrapolating linearly instead of clamping outside x.

        `x` must be strictly increasing.
        """
        x_new = np.asarray(x_new, dtype=float)
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        out = np.interp(x_new, x, y)
        below, above = x_new < x[0], x_new > x[-1]
        if below.any():
            slope = (y[1] - y[0]) / (x[1] - x[0])
            out[below] = y[0] + slope * (x_new[below] - x[0])
        if above.any():
            slope = (y[-1] - y[-2]) / (x[-1] - x[-2])
            out[above] = y[-1] + slope * (x_new[above] - x[-1])
        return out

    # Supercell base state: the U(z), V(z) we want to preserve.
    sc_bs_ds = xrr.read_rams_output(
        Path(
            "/gpfs/wolf2/arm/atm124/proj-shared/cmdavis4/projects/bl_transport/rams_io/sc_wk/paper_base_state/output"
        ).glob("a-A-*.h5")
    )
    sc_bs_ds = xrr.calculate_thermodynamic_variables(sc_bs_ds)
    sc_bs_profile = sc_bs_ds.mean(["x", "y"]).isel(time=0, drop=True).squeeze()
    sc_bs_z = sc_bs_profile["z"].values

    # Pressure levels of the new sounding (IPSFLG=0 -> PS really is pressure).
    current_ps = np.asarray(
        xrr.soundings._parse_ramsin_sounding_array(
            sl_ramsin_path.read_text(), "PS"
        ),
        dtype=float,
    )

    # Step 1: pressure -> height.  np.interp requires an increasing x, and
    # pressure decreases with height, so interpolate against -log(p), which is
    # both increasing and close to linear in z.
    neg_log_p = lambda p: -np.log(np.asarray(p, dtype=float))
    sounding_z = interp_with_linear_extrapolation(
        neg_log_p(current_ps), neg_log_p(sl_bs_p), sl_bs_z
    )
    assert np.all(np.diff(sounding_z) > 0), "sounding heights are not increasing"

    # Step 2: height -> wind.  Clamping at the ends is fine here: the supercell
    # wind profile is constant above ~10 km and the lowest sounding level is at
    # the ground.
    for var in ["US", "VS"]:
        ramsin_params[var] = xrr.to_ramsin_values_str(
            np.interp(sounding_z, sc_bs_z, sc_bs_profile[var[0] + "C"].values)
        )

    display(ramsin_params)

    ramsin_path, submit_script_path = xrr.execution.setup_rams_run(
        run_dir=experiment_dir,
        # run_name=experiment_subgroup,
        ramsin_parameters=ramsin_params,
        ramsin_template_path=sc_ramsin_template_path,
        render_kwargs=dict(
            account="atm124",
            modules=ps.machines.get_current_host().slurm.modules_for("rams"),
            mpi_launcher=ps.machines.get_current_host().slurm.mpi_launcher,
            n_nodes=N_NODES,
            n_cores=128 * N_NODES,
            wall_time="20:00:00",
            rams_executable_path=RAMS_EXECUTABLE_PATH,
            queue=SLURM_QUEUE,
        ),
        new_return_signature=True,
    )
    ramsins_paths[experiment_name] = ramsin_path

    if SUBMIT_SLURM_JOBS:
        subprocess.run(["sbatch", str(submit_script_path.resolve())])
    print("")

diff_ramsins_by_parameter(ramsins_paths)

In [ ]:
# Sanity check: the new sounding's U(z), V(z) should lie on top of the
# supercell base state it was taken from.
fig, axs = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
for ax, var in zip(axs, ["US", "VS"]):
    new_wind = np.array([
        float(v) for v in ramsin_params[var].replace("\n", "").split(",")
    ])
    ax.plot(
        sc_bs_profile[var[0] + "C"].values,
        sc_bs_z / 1000,
        lw=3,
        alpha=0.4,
        label="supercell base state",
    )
    ax.plot(new_wind, sounding_z / 1000, ls="--", label="new sounding")
    ax.set(xlabel=f"{var[0]} (m/s)", ylim=(0, 20))
axs[0].set_ylabel("z (km)")
axs[0].legend()
